In [ ]:
import torch
from SIM2EikonalSolverPyTorch import eikonal_solver, tesselate_volume
import matplotlib.pyplot as plt
from math import log
from scipy.io import savemat

device = "cuda"

In [ ]:
# shape = S, Or, H, W = 32, 64, 100, 100
# cost = torch.ones(*shape)

In [ ]:
# source_points = (
#     (S//2, Or//2, H//2, W//2),
# )

In [ ]:
# dθ = 2 * torch.pi / Or
# a_min = 0.25
# a_max = a_min**(-(S - 1 - S//2)/(S//2))
# τ_min, τ_max = log(a_min), log(a_max)
# scales = torch.exp(torch.linspace(τ_min, τ_max, S))
# ws = (1., 5., 25., 100.)

In [ ]:
# W, grad_W = eikonal_solver(cost, source_points, a_min, a_max, ws, n_max=5e3, initial_condition=1e4, device=device)

In [ ]:
# fig, ax = plt.subplots(2, 4, figsize=(24, 10), layout="constrained")
# for i, a in enumerate(ax.flat):
#     cbar = a.imshow(W[S//2, 8*i], cmap="gray")
#     fig.colorbar(cbar, ax=a)
#     a.set_title(fr"$\theta = {2 * (8 * i)/Or:.2f} \pi$")

In [ ]:
# fig, ax = plt.subplots(2, 4, figsize=(24, 10), layout="constrained")
# for i, a in enumerate(ax.flat):
#     cbar = a.imshow(W[4 * i, Or//2], cmap="gray")
#     fig.colorbar(cbar, ax=a)
#     a.set_title(fr"$a = {scales[4 * i]:.2f}$")

In [ ]:
# savemat("SIM2_ball.mat", {"ball": W})

In [ ]:
lie_algebra_basis = torch.tensor([
    [[0.,  0., 1.],
     [0.,  0., 0.],
     [0.,  0., 0.],],
    [[0.,  0., 0.],
     [0.,  0., 1.],
     [0.,  0., 0.],],
    [[0., -1., 0.],
     [1.,  0., 0.],
     [0.,  0., 0.],],
    [[1.,  0., 0.],
     [0.,  1., 0.],
     [0.,  0., 0.],],
])

def logarithmic_spiral(c, W, H, Or, S, a_min, a_max, N, t_start=-1, t_end=1):
    ts = torch.linspace(t_start, t_end, N)[:, None, None, None]
    gs = torch.matrix_exp(((ts * c[:, None, None]) * lie_algebra_basis).sum(1))
    # xs in [-1, 1]
    xs = gs[:, 0, 2]
    # ys in [-1, 1]
    ys = gs[:, 1, 2]
    # θs in [-π, π]
    θs = torch.atan2(gs[:, 1, 0], gs[:, 0, 0])
    # ass in [a_min, a_max]
    ass = (gs[:, 0, 0]**2 + gs[:, 1, 0]**2).sqrt()
    # to indices
    ws = torch.clamp(W//2 * (xs + 1), 0, W - 1)
    hs = torch.clamp(H//2 * (ys + 1), 0, H - 1)
    ors = torch.clamp((θs % (2 * torch.pi)) * Or / (2 * torch.pi), 0, Or - 1)
    ss = torch.clamp((torch.log(ass / a_min) / log(a_max / a_min)) * (S - 1), 0, S - 1)

    return torch.stack((ss, ors, hs, ws), dim=-1)

In [ ]:
shape = S, Or, H, W = 8, 16, 100, 100
a_min = 0.25
a_max = a_min**(-(S - 1 - S//2)/(S//2))
τ_min, τ_max = log(a_min), log(a_max)
scales = torch.exp(torch.linspace(τ_min, τ_max, S))
γ = logarithmic_spiral(torch.tensor([0.2, 0., 1., 0.3]), W, H, Or, S, a_min, a_max, 10, t_start=-2, t_end=6)
plt.plot(γ[:, -2], γ[:, -1])

In [ ]:

seeds = tuple(tuple(int(x) for x in p) for p in γ)
dθ = 2 * torch.pi / Or
a_min = 0.25
a_max = a_min**(-(S - 1 - S//2)/(S//2))
τ_min, τ_max = log(a_min), log(a_max)
scales = torch.exp(torch.linspace(τ_min, τ_max, S))
ws = (1., 3., 15., 50.)

tesselated = tesselate_volume(shape, seeds, a_min, a_max, ws, n_distance_max=5e3, device=device).cpu()

In [ ]:
index = 0
a, θ, y, x = seeds[index]
for i in range(-1, 2):
    for j in range(-1, 2):
        for k in range(-1, 2):
            for l in range(-1, 2):
                print(tesselated[a + i, θ + j, y + k, x + l])

In [ ]:
plt.imshow(tesselated.max(0)[0].max(0)[0])
plt.plot(γ[:, -2], γ[:, -1])